# ODEON × I-Simpa × Pyroomacoustics — Colab, diagnóstico de convergência v2

Preserva as dez superfícies do `Example.par`, a fonte OMNI e os 25 receptores. Esta versão registra **configurações e RIRs brutas**, de modo que os resultados possam ser auditados.

**Roteiro:** visualizar a geometria → executar 3.000 raios para diagnóstico → inspecionar curvas de Schroeder → exportar RIRs e métricas → aumentar para 15.000 e 50.000 raios em novas execuções. Os resultados de cada execução devem ser baixados antes de aumentar os parâmetros.

**Atenção:** a opção inicial é de diagnóstico, e não um resultado RT30 validado. O Colab pode levar vários minutos na etapa de cálculo acústico.


## 1. Preparar e visualizar a geometria — sem instalar Pyroomacoustics

In [ ]:
!pip -q install plotly pandas numpy scipy

In [ ]:
RECEIVER_ROWS = [('R01', 8.0, -5.0, 1.52), ('R02', 8.0, -2.0, 1.52), ('R03', 8.0, 1.0, 1.52), ('R04', 8.0, 4.0, 1.52), ('R05', 8.0, 7.0, 1.52), ('R06', 10.962, -5.0, 1.994), ('R07', 10.962, -2.0, 1.994), ('R08', 10.962, 1.0, 1.994), ('R09', 10.962, 4.0, 1.994), ('R10', 10.962, 7.0, 1.994), ('R11', 13.925, -5.0, 2.468), ('R12', 13.925, -2.0, 2.468), ('R13', 13.925, 1.0, 2.468), ('R14', 13.925, 4.0, 2.468), ('R15', 13.925, 7.0, 2.468), ('R16', 16.887, -5.0, 2.942), ('R17', 16.887, -2.0, 2.942), ('R18', 16.887, 1.0, 2.942), ('R19', 16.887, 4.0, 2.942), ('R20', 16.887, 7.0, 2.942), ('R21', 19.849, -5.0, 3.416), ('R22', 19.849, -2.0, 3.416), ('R23', 19.849, 1.0, 3.416), ('R24', 19.849, 4.0, 3.416), ('R25', 19.849, 7.0, 3.416)]
# Geometria da sala Example.par de ODEON (10 superfícies físicas)
# Comprimento 22 m, largura no palco 10 m, largura na plateia 16 m.
import numpy as np
import pandas as pd
import plotly.graph_objects as go

VERTICES_POS = {
 101: (0.,5.,0.), 102: (6.,8.,0.), 103: (22.,8.,2.56),
 201: (0.,5.,7.5), 202: (6.,8.,10.), 203: (22.,8.,10.)
}
P = {idx: np.array(xyz,dtype=float) for idx,xyz in VERTICES_POS.items()}
for idx, xyz in list(P.items()):
    P[-idx] = xyz*np.array([1.,-1.,1.])

# A ordem abaixo corresponde às superfícies nomeadas no PAR de ODEON.
FACES = {
    '1001 Piso palco': [102,-102,-101,101],
    '1002 Piso plateia': [102,103,-103,-102],
    '2001 Parede frontal': [201,-201,-101,101],
    '2002 Parede palco norte': [101,201,202,102],
    '2002 Parede palco sul': [-101,-102,-202,-201],
    '2003 Parede plateia norte': [102,103,203,202],
    '2003 Parede plateia sul': [-102,-202,-203,-103],
    '2004 Parede traseira': [103,203,-203,-103],
    '3001 Teto palco': [202,-202,-201,201],
    '3002 Teto plateia': [202,203,-203,-202],
}

# Normal exterior obrigatória para o modelo de paredes do Pyroomacoustics.
INTERIOR = np.array([10.,0.,5.])
def outward_polygon(ids):
    pts=np.array([P[i] for i in ids],float)
    normal=np.cross(pts[1]-pts[0],pts[2]-pts[0])
    assert np.linalg.norm(normal)>0
    if np.dot(normal,INTERIOR-pts.mean(axis=0))>0:
        pts=pts[::-1]
    return pts
WALL_POLYS={name:outward_polygon(ids) for name,ids in FACES.items()}

# Coordenadas medidas diretamente do config.xml I-Simpa enviado (25 receptores)
RECEIVERS = pd.DataFrame(RECEIVER_ROWS, columns=['receptor','x_m','y_m','z_m'])
SRC = np.array([3.,0.,4.])
R = RECEIVERS[['x_m','y_m','z_m']].to_numpy().T
assert len(RECEIVERS)==25 and len(WALL_POLYS)==10
for _,row in RECEIVERS.iterrows():
    floor = max(0, (row['x_m']-6)*0.16)
    assert row['z_m']>floor and row['z_m']<10, f"Microfone fora: {row['receptor']}"

print(f'OK: {len(WALL_POLYS)} superfícies, {R.shape[1]} receptores. Fonte: {SRC}.')


In [ ]:
# Visualização interativa sem precisar de Pyroomacoustics
COLORS={'Piso':'#64b6ac','Teto':'#94b8d7','Parede':'#b4b5c3'}
fig=go.Figure()
for name,pts in WALL_POLYS.items():
    kind='Piso' if 'Piso' in name else ('Teto' if 'Teto' in name else 'Parede')
    pts4=pts
    # triangulação dos quadriláteros
    i,j,k=[0,0],[1,2],[2,3]
    fig.add_trace(go.Mesh3d(x=pts4[:,0],y=pts4[:,1],z=pts4[:,2],
        i=i,j=j,k=k,flatshading=True,color=COLORS[kind],opacity=.22,
        name=name,hovertemplate=name+'<extra></extra>',showscale=False))
    pts5=np.vstack([pts4,pts4[0]])
    fig.add_trace(go.Scatter3d(x=pts5[:,0],y=pts5[:,1],z=pts5[:,2],mode='lines',
        line=dict(color='#4d6373',width=4),showlegend=False,hoverinfo='skip'))
fig.add_trace(go.Scatter3d(x=R[0],y=R[1],z=R[2],mode='markers+text',
    marker=dict(size=4,color='#156ec0'),text=RECEIVERS['receptor'],textposition='top center',
    name='R01–R25',hovertemplate='%{text}: (%{x:.3f}, %{y:.3f}, %{z:.3f}) m<extra></extra>'))
fig.add_trace(go.Scatter3d(x=[SRC[0]],y=[SRC[1]],z=[SRC[2]],mode='markers+text',
    marker=dict(size=9,color='#d94a47',symbol='diamond'),text=['P1 OMNI'],
    textposition='top center',name='Fonte P1'))
fig.update_layout(title='Auditório ODEON Example.par — geometria exata + 25 receptores',
    scene=dict(xaxis_title='X (m)',yaxis_title='Y (m)',zaxis_title='Z (m)',
        aspectmode='data',camera=dict(eye=dict(x=1.45,y=-1.6,z=.9))),
    margin=dict(l=0,r=0,t=60,b=0),height=620,legend=dict(orientation='h'))
fig.show()
fig.write_html('auditorio_odeon_interativo.html',include_plotlyjs='cdn')
print('Visualização HTML salva: auditorio_odeon_interativo.html')


**Confirmação visual:** inspecione o teto mais baixo sobre o palco, a expansão lateral a partir de x=0→6 m, a inclinação do piso da plateia e a linha dos 25 receptores a aproximadamente 1,2 m acima desse piso.

In [ ]:
# Exportar geometria OBJ e PLY (opcional, para conferência e reutilização)
vertex_ids=sorted(P,key=lambda k:(P[k][0],P[k][1],P[k][2]))
coords=[]
for idx in vertex_ids:
    v=tuple(np.round(P[idx],9))
    if v not in coords: coords.append(v)
vmap={v:idx+1 for idx,v in enumerate(coords)}
faces_idx=[]
for name,poly in WALL_POLYS.items():
    faces_idx.append((name,[vmap[tuple(np.round(q,9))] for q in poly]))
with open('auditorio_odeon_10_superficies.obj','w') as fh:
    fh.write('# ODEON Example.par, coordinates in metres\n')
    for x,y,z in coords: fh.write(f'v {x} {y} {z}\n')
    for name,ii in faces_idx:
        fh.write('g '+name.replace(' ','_')+'\n')
        fh.write('f '+' '.join(map(str,ii))+'\n')
with open('auditorio_odeon_10_superficies.ply','w') as fh:
    fh.write('ply\nformat ascii 1.0\ncomment units metres\n')
    fh.write(f'element vertex {len(coords)}\nproperty float x\nproperty float y\nproperty float z\n')
    fh.write(f'element face {len(faces_idx)}\nproperty list uchar int vertex_indices\nend_header\n')
    for x,y,z in coords:fh.write(f'{x} {y} {z}\n')
    for name,ids in faces_idx:fh.write(str(len(ids))+' '+' '.join(str(i-1) for i in ids)+'\n')
RECEIVERS.to_csv('receptores_25.csv',index=False)
print('Exportados: OBJ, PLY, receptores_25.csv')


## 2. Conferir a referência ODEON e I-Simpa de 1000 Hz

As tabelas ODEON (oitava) e I-Simpa (soma de energias das bandas de terço 800/1000/1250 Hz) foram incorporadas ao notebook para comparação. R11 tinha parâmetros derivados incompletos na extração original: mantenha **NaN**, não preencha valores artificiais.

In [ ]:
import io
REF_CSV = 'receptor,x_m,y_m,z_m,SPL_iSimpa_800_Hz_dB,SPL_iSimpa_1000_Hz_dB,SPL_iSimpa_1250_Hz_dB,SPL_iSimpa_octava_1000_Hz_dB,G_iSimpa_dB,G_ODEON_dB,delta_G_dB,C80_iSimpa_dB,C80_ODEON_dB,delta_C80_dB,EDT_iSimpa_s,EDT_ODEON_s,delta_EDT_s,T30_iSimpa_s,T30_ODEON_s,delta_T30_s,estado_parametros\nR01,8.000000,-5.000000,1.520000,50.198000,50.263000,50.239000,55.005000,15.398000,14.700000,0.698000,-3.291000,-2.100000,-1.191000,3.735000,3.380000,0.355000,3.741000,3.330000,0.411000,ok\nR02,8.000000,-2.000000,1.520000,50.349000,50.237000,50.266000,55.056000,15.449000,14.800000,0.649000,-3.753000,-2.300000,-1.453000,3.732000,3.380000,0.352000,3.723000,3.330000,0.393000,ok\nR03,8.000000,1.000000,1.520000,50.251000,50.192000,50.137000,54.965000,15.358000,14.900000,0.458000,-3.826000,-2.500000,-1.326000,3.782000,3.330000,0.452000,3.720000,3.310000,0.410000,ok\nR04,8.000000,4.000000,1.520000,50.269000,50.165000,50.267000,55.005000,15.398000,14.800000,0.598000,-3.480000,-2.300000,-1.180000,3.746000,3.400000,0.346000,3.724000,3.320000,0.404000,ok\nR05,8.000000,7.000000,1.520000,50.388000,50.453000,50.460000,55.205000,15.598000,15.000000,0.598000,-2.752000,-2.000000,-0.752000,3.628000,3.350000,0.278000,3.716000,3.330000,0.386000,ok\nR06,10.962000,-5.000000,1.994000,50.300000,50.228000,50.264000,55.035000,15.428000,14.700000,0.728000,-3.549000,-2.400000,-1.149000,3.669000,3.370000,0.299000,3.735000,3.300000,0.435000,ok\nR07,10.962000,-2.000000,1.994000,50.158000,50.250000,50.110000,54.944000,15.337000,14.700000,0.637000,-4.312000,-3.000000,-1.312000,3.727000,3.280000,0.447000,3.719000,3.330000,0.389000,ok\nR08,10.962000,1.000000,1.994000,50.182000,50.204000,50.216000,54.972000,15.365000,14.800000,0.565000,-4.177000,-3.000000,-1.177000,3.710000,3.240000,0.470000,3.716000,3.310000,0.406000,ok\nR09,10.962000,4.000000,1.994000,50.321000,50.304000,50.232000,55.057000,15.450000,14.700000,0.750000,-3.566000,-2.400000,-1.166000,3.713000,3.290000,0.423000,3.737000,3.330000,0.407000,ok\nR10,10.962000,7.000000,1.994000,50.129000,50.128000,50.178000,54.916000,15.309000,14.700000,0.609000,-3.533000,-2.700000,-0.833000,3.656000,3.300000,0.356000,3.703000,3.320000,0.383000,ok\nR11,13.925000,-5.000000,2.468000,50.542000,50.529000,50.481000,55.289000,15.682000,14.700000,0.982000,,-2.100000,,,3.330000,,,3.280000,,ausentes; ecograma presente\nR12,13.925000,-2.000000,2.468000,50.406000,50.294000,50.408000,55.141000,15.534000,14.700000,0.834000,-3.615000,-2.400000,-1.215000,3.661000,3.240000,0.421000,3.761000,3.310000,0.451000,ok\nR13,13.925000,1.000000,2.468000,50.445000,50.513000,50.491000,55.254000,15.647000,14.700000,0.947000,-3.541000,-2.600000,-0.941000,3.640000,3.240000,0.400000,3.763000,3.340000,0.423000,ok\nR14,13.925000,4.000000,2.468000,50.504000,50.479000,50.525000,55.274000,15.667000,14.800000,0.867000,-3.300000,-2.300000,-1.000000,3.614000,3.170000,0.444000,3.758000,3.330000,0.428000,ok\nR15,13.925000,7.000000,2.468000,50.409000,50.514000,50.567000,55.268000,15.661000,14.900000,0.761000,-3.222000,-2.300000,-0.922000,3.602000,3.240000,0.362000,3.729000,3.320000,0.409000,ok\nR16,16.887000,-5.000000,2.942000,50.523000,50.598000,50.539000,55.325000,15.718000,14.900000,0.818000,-2.698000,-1.800000,-0.898000,3.608000,3.220000,0.388000,3.778000,3.300000,0.478000,ok\nR17,16.887000,-2.000000,2.942000,50.492000,50.441000,50.388000,55.212000,15.605000,14.700000,0.905000,-2.835000,-1.700000,-1.135000,3.636000,3.210000,0.426000,3.801000,3.330000,0.471000,ok\nR18,16.887000,1.000000,2.942000,50.482000,50.604000,50.497000,55.300000,15.693000,14.900000,0.793000,-3.674000,-1.900000,-1.774000,3.580000,3.150000,0.430000,3.786000,3.290000,0.496000,ok\nR19,16.887000,4.000000,2.942000,50.632000,50.668000,50.636000,55.417000,15.810000,14.800000,1.010000,-2.574000,-1.500000,-1.074000,3.568000,3.200000,0.368000,3.780000,3.320000,0.460000,ok\nR20,16.887000,7.000000,2.942000,50.477000,50.394000,50.383000,55.189000,15.582000,14.900000,0.682000,-2.527000,-1.500000,-1.027000,3.636000,3.180000,0.456000,3.763000,3.330000,0.433000,ok\nR21,19.849000,-5.000000,3.416000,50.390000,50.402000,50.387000,55.164000,15.557000,14.800000,0.757000,-2.447000,-1.500000,-0.947000,3.713000,3.310000,0.403000,3.803000,3.290000,0.513000,ok\nR22,19.849000,-2.000000,3.416000,50.476000,50.550000,50.430000,55.257000,15.650000,14.800000,0.850000,-3.203000,-1.200000,-2.003000,3.619000,3.240000,0.379000,3.792000,3.300000,0.492000,ok\nR23,19.849000,1.000000,3.416000,50.539000,50.497000,50.538000,55.296000,15.689000,15.000000,0.689000,-3.120000,-1.400000,-1.720000,3.605000,3.160000,0.445000,3.792000,3.310000,0.482000,ok\nR24,19.849000,4.000000,3.416000,50.506000,50.430000,50.641000,55.298000,15.691000,14.800000,0.891000,-3.125000,-1.400000,-1.725000,3.631000,3.130000,0.501000,3.796000,3.330000,0.466000,ok\nR25,19.849000,7.000000,3.416000,50.274000,50.343000,50.389000,55.107000,15.500000,14.500000,1.000000,-2.273000,-1.000000,-1.273000,3.668000,3.340000,0.328000,3.773000,3.340000,0.433000,ok\n'
ref = pd.read_csv(io.StringIO(REF_CSV))
print('Receptores:',len(ref))
print('Colunas:',list(ref.columns))
display(ref[['receptor','SPL_iSimpa_octava_1000_Hz_dB','C80_ODEON_dB','C80_iSimpa_dB','EDT_ODEON_s','EDT_iSimpa_s','T30_ODEON_s','T30_iSimpa_s']].head(10))

## 3. Instalar Pyroomacoustics somente quando a geometria estiver aprovada

Em Colab, a instalação pode compilar a extensão C++ e consumir alguns minutos. A documentação do projeto recomenda memória disponível de aproximadamente 4 GB ou mais para builds. A célula abaixo não instala nada no computador local. <https://github.com/LCAV/pyroomacoustics>

In [ ]:
import sys, subprocess, os
os.environ.setdefault('CMAKE_BUILD_PARALLEL_LEVEL','2')
os.environ.setdefault('MAX_JOBS','2')
subprocess.check_call([sys.executable,'-m','pip','install','-q','pyroomacoustics==0.10.1'])
import pyroomacoustics as pra
print('Pyroomacoustics:',pra.__version__)

## 4. Criar a sala 3D real no Pyroomacoustics

Não usar `pra.ShoeBox` para o benchmark principal: perderíamos a geometria inclinada. Em vez disso, criamos objetos `Wall` para as dez faces e depois `pra.Room(walls, ...)`. Verificamos a posição de todos os receptores.

In [ ]:
import pyroomacoustics as pra
FS = 8000
ALPHA = 0.10
SCATTERING = 0.0
MAX_ORDER = 2
N_RAYS = 3000                # primeira prova; depois repetir com 15000 / 50000
RAY_TMAX = 8.0               # máximo de voo dos raios, não garante RT30 válido
ENERGY_THRES = 1e-7
RADIUS = 0.31
HIST_BIN_S = 0.010
SEED = 123                    # rótulo no protocolo; o RNG do núcleo C++ pode diferir
np.random.seed(SEED)
walls = [pra.wall_factory(poly.T.copy(), np.array([ALPHA]),
          np.array([SCATTERING]), name) for name,poly in WALL_POLYS.items()]
room = pra.Room(walls, fs=FS, max_order=MAX_ORDER, air_absorption=False)
assert room.dim == 3 and room.is_inside(SRC)
for _,rr in RECEIVERS.iterrows():
    assert room.is_inside(rr[['x_m','y_m','z_m']].to_numpy(dtype=float)), rr.receptor
room.add_source(SRC)
room.add_microphone_array(R)
print('Sala:',len(room.walls),'paredes;',room.mic_array.nmic,'receptores')


## 5. Simular respostas impulsivas (ISM + ray tracing)

Primeiro experimente com `N_RAYS = 750`. Se a simulação rodar normalmente, aumente gradualmente. **Não compare quantidade de partículas de I-Simpa com quantidade de raios de PRA como se fossem equivalentes.** A API Pyroomacoustics usa estimativa energética por *ray tracing* somada ao modelo de fontes imagem. Consulte <https://pyroomacoustics.readthedocs.io/en/pypi-release/pyroomacoustics.room.html>.

In [ ]:
import time, json
from pathlib import Path
start = time.perf_counter()
room.set_ray_tracing(n_rays=N_RAYS, receiver_radius=RADIUS,
                     energy_thres=ENERGY_THRES, time_thres=RAY_TMAX,
                     hist_bin_size=HIST_BIN_S)
room.compute_rir()
elapsed = time.perf_counter()-start
print(f'compute_rir: {elapsed:.2f} segundos | N_RAYS={N_RAYS} | RT limite={RAY_TMAX}s')
assert len(room.rir)==25
# Guardar a lista de RIR, sem preencher artificialmente até uma duração fixa.
rirs = {f'R{i+1:02d}':np.asarray(room.rir[i][0], dtype=np.float32)
        for i in range(len(room.rir))}
np.savez_compressed('rirs_25.npz', **rirs)
config = {'pyroom_version':pra.__version__,'fs_hz':FS,'absorption':ALPHA,
          'scattering':SCATTERING,'max_order_ism':MAX_ORDER,'n_rays':N_RAYS,
          'ray_tmax_s':RAY_TMAX,'energy_threshold':ENERGY_THRES,
          'receiver_radius_m':RADIUS,'histogram_bin_s':HIST_BIN_S,
          'air_absorption':False,'n_receptors':len(rirs),
          'source_m':SRC.tolist(),'seed_label':SEED,
          'compute_rir_seconds':elapsed,
          'rir_duration_s':{key:len(val)/FS for key,val in rirs.items()}}
Path('configuracao_pyroom.json').write_text(json.dumps(config,indent=2),encoding='utf-8')
print('RIRs e parâmetros de execução exportados para auditoria.')


## 6. Calcular EDT, T30, C80 e níveis espaciais na banda de oitava de 1000 Hz

Aplicamos filtro digital Butterworth na faixa **707–1414 Hz**. As definições são de referência, não reprodução bit a bit do pós-processamento ODEON/I-Simpa. EDT resulta do ajuste 0 a −10 dB; T30, de −5 a −35 dB na curva de Schroeder. Quando o intervalo não é obtido, devolvemos `NaN`. Para C80 separamos a energia antes/depois de 80 ms, contados desde a chegada direta teórica (distância/c).

In [ ]:
from scipy.signal import butter, sosfiltfilt
from scipy.stats import linregress
import matplotlib.pyplot as plt
C=float(room.c)
sos=butter(4,[1000/np.sqrt(2),1000*np.sqrt(2)],btype='bandpass',fs=FS,output='sos')

def decay_analysis(h,mic):
    h=np.asarray(h,dtype=float)
    if h.size < 100 or np.sum(h*h)==0: return None
    hb=sosfiltfilt(sos,h)
    eng=hb*hb
    t=np.arange(eng.size)/FS
    edc=np.cumsum(eng[::-1])[::-1]
    if edc[0]<=0: return None
    db=10*np.log10(np.maximum(edc,1e-300)/edc[0])
    arrival=np.linalg.norm(mic-SRC)/C
    idx80=np.searchsorted(t,arrival+0.080)
    early=eng[:idx80].sum(); late=eng[idx80:].sum()
    c80=10*np.log10(early/late) if early>0 and late>0 else np.nan
    # EDC é normalizada em 0 dB no início. Regressão apenas como diagnóstico.
    def regress(upper,lower):
        ix=np.where((db<=upper)&(db>=lower))[0]
        if ix.size < 10 or np.ptp(db[ix]) < .9*(upper-lower):
            return np.nan,np.nan
        reg=linregress(t[ix],db[ix])
        if reg.slope >= 0: return np.nan,reg.rvalue**2
        return -60/reg.slope,reg.rvalue**2
    edt,r2_edt=regress(0,-10)
    t30,r2_t30=regress(-5,-35)
    # energia contida nos últimos 5% da resposta calculada
    tail_frac=float(eng[int(.95*len(eng)):].sum()/eng.sum())
    return {'e':eng,'t':t,'sch_db':db,'relative_energy_db':10*np.log10(eng.sum()),
            'C80_PRA_dB':c80,'EDT_PRA_s':edt,'T30_PRA_s':t30,
            'R2_EDT':r2_edt,'R2_T30':r2_t30,
            'tail_energy_fraction':tail_frac,'rir_duration_s':len(h)/FS,
            'distance_m':np.linalg.norm(mic-SRC)}

rows=[]; debug={}
for i,rr in RECEIVERS.iterrows():
    a=decay_analysis(room.rir[i][0],R[:,i]);debug[rr.receptor]=a
    if a is None: continue
    rows.append([rr.receptor,rr.x_m,rr.y_m,rr.z_m,
                 a['relative_energy_db'],a['C80_PRA_dB'],a['EDT_PRA_s'],
                 a['T30_PRA_s'],a['distance_m'],a['R2_T30'],a['tail_energy_fraction'],
                 a['rir_duration_s']])
pra_df=pd.DataFrame(rows, columns=['receptor','x_m','y_m','z_m','energia_rel_dB',
             'C80_PRA_dB','EDT_PRA_s','T30_PRA_s','distancia_m',
             'R2_T30','fracao_energia_cauda_5pct','duracao_RIR_s'])
pra_df['SPL_PRA_centrado_dB']=pra_df['energia_rel_dB']-pra_df['energia_rel_dB'].mean()
pra_df['flag_T30_revisar']=(pra_df['T30_PRA_s']>6)|((pra_df['R2_T30']<.97)&pra_df['R2_T30'].notna())
print('ATENÇÃO: valores T30 desta etapa ainda são preliminares, não validados.')
print(pra_df[['receptor','energia_rel_dB','C80_PRA_dB','T30_PRA_s','R2_T30','flag_T30_revisar']].to_string(index=False))
pra_df.to_csv('pyroom_25_receptores_octava_1000Hz.csv',index=False)


## 7. Comparação **preliminar**, sem tratar RT30 como validado
RT30 precisa de número de raios suficiente, curva de Schroeder com faixa de decaimento mensurável e análise da cauda. Valores muito superiores a 3–4 s no cenário deste auditório deverão ser investigados antes de conclusões.


In [ ]:
compare=ref.merge(pra_df,on='receptor',how='inner',suffixes=('_ref','_pra'))
compare['SPL_iSimpa_centrado_dB']=(compare['SPL_iSimpa_octava_1000_Hz_dB']-compare['SPL_iSimpa_octava_1000_Hz_dB'].mean())
compare['delta_SPL_rel_PRA_menos_iSimpa_dB']=(compare['SPL_PRA_centrado_dB']-compare['SPL_iSimpa_centrado_dB'])
for k,refcol in [('C80','C80_ODEON_dB'),('EDT','EDT_ODEON_s'),('T30','T30_ODEON_s')]:
    compare['delta_'+k+'_PRA_menos_ODEON']=compare[k+'_PRA_'+('dB' if k=='C80' else 's')]-compare[refcol]
cols=['receptor','SPL_iSimpa_centrado_dB','SPL_PRA_centrado_dB',
      'C80_ODEON_dB','C80_iSimpa_dB','C80_PRA_dB',
      'T30_ODEON_s','T30_iSimpa_s','T30_PRA_s']
display(compare[cols])

for k in ['C80','EDT','T30']:
    d=compare['delta_'+k+'_PRA_menos_ODEON'].dropna().to_numpy()
    if len(d): print(f'{k}: N={len(d)}, bias={np.mean(d):+.3f}, RMSE={np.sqrt(np.mean(d*d)):.3f}')
print('SPL espacial relativo: RMSE (PRA vs I-Simpa) =',
      round(float(np.sqrt(np.mean(compare['delta_SPL_rel_PRA_menos_iSimpa_dB']**2))),3),'dB')
compare.to_csv('comparacao_ODEON_ISimpa_Pyroom_25.csv',index=False)


## 8. Visualizar mapa de diferenças e resultados

In [ ]:
import plotly.express as px
fig_spl=px.scatter(compare,x='x_m_ref',y='y_m_ref',
    color='delta_SPL_rel_PRA_menos_iSimpa_dB',text='receptor',
    color_continuous_scale='RdBu_r',
    title='PRA − I-Simpa: diferença de SPL relativo na oitava 1000 Hz',
    labels={'x_m_ref':'X (m)','y_m_ref':'Y (m)',
            'delta_SPL_rel_PRA_menos_iSimpa_dB':'Δ SPL relativo (dB)'})
fig_spl.update_traces(marker=dict(size=24),textposition='top center')
fig_spl.update_layout(yaxis=dict(scaleanchor='x',scaleratio=1),height=540)
fig_spl.show()

fig_t=go.Figure()
for col,name in [('T30_ODEON_s','ODEON'),('T30_iSimpa_s','I-Simpa'),('T30_PRA_s','Pyroomacoustics')]:
    fig_t.add_trace(go.Scatter(x=compare.receptor,y=compare[col],mode='lines+markers',name=name))
fig_t.update_layout(title='T30 por receptor — oitava 1000 Hz',xaxis_title='Receptor',yaxis_title='T30 (s)')
fig_t.show()


In [ ]:
# Curvas de Schroeder e ecogramas para auditoria visual
sel=['R01','R13','R23']
fig,axes=plt.subplots(len(sel),1,figsize=(10,8),sharex=False)
for ax,rid in zip(axes,sel):
    d=debug.get(rid)
    if not d: continue
    ax.plot(d['t'],d['sch_db'],label=rid)
    ax.axhline(-5,ls='--',lw=.8)
    ax.axhline(-35,ls='--',lw=.8)
    ax.set_ylim(-65,5)
    ax.set_xlim(0,min(RAY_TMAX,d['t'][-1]))
    ax.set_ylabel('Schroeder (dB)')
    ax.legend(loc='upper right')
    ax.grid(alpha=.2)
axes[-1].set_xlabel('Tempo (s)')
fig.suptitle('Diagnóstico de decaimento — observar ruído e truncamento da cauda')
plt.tight_layout();plt.savefig('schroeder_diagnostico.png',dpi=150);plt.show()


## 9. Exportar os resultados

Arquivos gerados nesta execução:

- `auditorio_odeon_interativo.html`: modelo 3D manipulável;
- `auditorio_odeon_10_superficies.obj` / `.ply`: malhas da geometria;
- `receptores_25.csv`: posições;
- `pyroom_25_receptores_octava_1000Hz.csv`: resultados Pyroomacoustics;
- `comparacao_ODEON_ISimpa_Pyroom_25.csv`: os três simuladores.

Para baixar diretamente do Colab, rode a célula abaixo. **Os números produzidos serão resultados preliminares do modelo híbrido ISM + RT do PRA**, ainda não um benchmark completamente harmonizado.


In [ ]:
from google.colab import files
import os,zipfile
names=['auditorio_odeon_interativo.html','auditorio_odeon_10_superficies.obj',
       'auditorio_odeon_10_superficies.ply','receptores_25.csv',
       'pyroom_25_receptores_octava_1000Hz.csv',
       'comparacao_ODEON_ISimpa_Pyroom_25.csv',
       'rirs_25.npz','configuracao_pyroom.json','schroeder_diagnostico.png']
with zipfile.ZipFile(f'pyroom_diagnostico_{N_RAYS}_raios.zip','w',zipfile.ZIP_DEFLATED) as z:
    for n in names:
        if os.path.exists(n):z.write(n)
files.download(f'pyroom_diagnostico_{N_RAYS}_raios.zip')


## Sugestão de protocolo para convergência

1. Rodar 3.000 raios com 8 s, baixar o ZIP com RIRs e configurações.
2. Ajustar `N_RAYS=15000`, recriar a sala executando desde a célula 12, baixar o novo ZIP.
3. Se a evolução for significativa, testar 50.000 raios. A execução pode ser longa no Colab.
4. Não confundir quantidade de raios Pyroom com partículas I-Simpa: são métodos distintos.
5. Analisar RT30 somente depois de avaliar as curvas temporais, extensão da cauda e estabilidade entre resoluções.
6. Não comparar o valor de energia relativa da RIR como SPL absoluto: requer calibração da fonte e da pressão.
